# HW2-1 单模型版本

Kaggle: `ml2021spring-hw2` · **public 0.74719 / private 0.74657**

配方（每一条都是实验换来的，理由见 `README.md`）：

| 组件 | 选择 | 为什么 |
|---|---|---|
| 输入 | k=2 窗口，5×429 = 2145 维 | 数据是 10 ms 帧，拼邻帧能补上下文；k=2 比 k=1/k=3 都好 |
| 架构 | `2145→1024→512→128→39`，Linear+BN+ReLU+Dropout 0.2 | 2.8 M 参数是容量拐点，再大就背题（v5 到 12.8 M 时 train 0.997） |
| 优化器 | AdamW lr 1.5e-3 + OneCycle，30 epoch | BN 让 lr 能上 1e-3 量级；OneCycle 省掉手调 epoch |
| 数据 | **100%**，不切验证集 | epoch 数已在上一步确定，留 20% 不用等于白扔 25 万帧 |
| 推理 | TTA：窗口偏移 −1/0/+1 取平均 | 边界帧的上下文有偏，平移能抹平一部分 |

跑完输出 `submission.csv` 和 `probs.npy`（后者给集成用）。


In [ ]:
import os, glob, time, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
DATA_DIR = os.path.dirname(glob.glob('/kaggle/input/**/train_11.npy', recursive=True)[0])
SUB_PATH = glob.glob('/kaggle/input/**/sampleSubmission.csv', recursive=True)[0]

lab = np.load(os.path.join(DATA_DIR, 'train_label_11.npy'))
if lab.dtype.kind in 'US':          # 这份数据的标签存成了字符串 '0'..'38'
    lab = lab.astype(np.int64)
train = np.load(os.path.join(DATA_DIR, 'train_11.npy'), mmap_mode='r')   # float64, 4.2 GB -> 用 mmap
test = np.load(os.path.join(DATA_DIR, 'test_11.npy'), mmap_mode='r')
print('train', train.shape, train.dtype, '| test', test.shape, '| labels', lab.shape, lab.dtype)


In [ ]:
K = 2                       # 上下文：t-K .. t+K
IN_DIM = 429 * (2*K + 1)    # 2145
BATCH = 512
NUM_EPOCH = 30
LR = 1.5e-3
SEED = 11

class WindowDataset(Dataset):
    """Returns (2K+1)*429 floats for frame i.

    拼接放在 __getitem__ 里做：预生成要 1.23M x 2145 x 4B = 10.5 GB，会爆内存。
    """
    def __init__(self, X, y=None, k=K, offset=0):
        self.X, self.y, self.k, self.offset = X, y, k, offset
        self.N = X.shape[0]
    def __len__(self):
        return self.N
    def __getitem__(self, i):
        c = i + self.offset
        idx = np.clip(np.arange(c-self.k, c+self.k+1), 0, self.N-1)   # 边界复制
        x = np.asarray(self.X[idx], dtype=np.float32).reshape(-1)
        if self.y is None:
            return torch.from_numpy(x)
        return torch.from_numpy(x), int(self.y[i])

train_loader = DataLoader(WindowDataset(train, lab), batch_size=BATCH,
                          shuffle=True, num_workers=4, pin_memory=True, drop_last=True)
print('in_dim', IN_DIM, '| steps/epoch', len(train_loader))


In [ ]:
class Classifier(nn.Module):
    def __init__(self, in_dim=IN_DIM, h=1024, drop=0.2):
        super().__init__()
        def block(i, o):
            return nn.Sequential(nn.Linear(i, o), nn.BatchNorm1d(o), nn.ReLU(), nn.Dropout(drop))
        self.net = nn.Sequential(block(in_dim, h), block(h, h//2), block(h//2, h//8), nn.Linear(h//8, 39))
    def forward(self, x):
        return self.net(x)


In [ ]:
def same_seeds(s):
    torch.manual_seed(s); np.random.seed(s); random.seed(s)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

same_seeds(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = Classifier().to(device)
criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=LR, total_steps=NUM_EPOCH*len(train_loader), pct_start=0.15)
print('params', sum(p.numel() for p in model.parameters()))

for epoch in range(NUM_EPOCH):
    model.train(); ok = n = 0; t0 = time.time()
    for xb, yb in train_loader:
        xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
        optimizer.zero_grad()
        out = model(xb)
        loss = criterion(out, yb)
        loss.backward(); optimizer.step(); scheduler.step()
        ok += (out.argmax(1) == yb).sum().item(); n += len(yb)
    print(f'[{epoch+1:03d}/{NUM_EPOCH}] train acc {ok/n:.5f} ({time.time()-t0:.0f}s)', flush=True)

torch.save(model.state_dict(), 'model.ckpt')
print('没有验证集，所以只能看 train acc —— 这一点在 README 里解释过')


In [ ]:
model.eval()
probs = None
for off in (-1, 0, 1):                       # TTA
    loader = DataLoader(WindowDataset(test, offset=off), batch_size=2048,
                        shuffle=False, num_workers=4, pin_memory=True)
    chunks = []
    with torch.no_grad():
        for xb in loader:
            chunks.append(F.softmax(model(xb.to(device, non_blocking=True)), 1).cpu().numpy())
    acc = np.concatenate(chunks)
    probs = acc if probs is None else probs + acc
    print('offset', off, 'done', flush=True)

probs = (probs / 3).astype(np.float16)
np.save('probs.npy', probs)
pred = probs.argmax(1)
print('pred', pred.shape, '| classes used', np.unique(pred).size, '/ 39')


In [ ]:
import pandas as pd
sub = pd.read_csv(SUB_PATH)
col = sub.columns[-1]
assert len(pred) == len(sub), (len(pred), len(sub))
sub[col] = pred
sub.to_csv('submission.csv', index=False)
print(sub.head()); print('rows', len(sub))
